# Compare two versions of a support agent in Prototype

Ridgeline Outfitters routes damaged gear to its repairs mailbox. Its form accepts only an email address, not a full sentence. Compare two instructions on the same customer question. Record each real model call as a Prototype span, check the stored email-format result, and decide which reply fits the form. The addresses are fictional; nothing sends email or changes live traffic.

## Install and connect

Use Python 3.12. Your OpenAI key pays for two model calls; your Future AGI keys record the experiment and run a format check. On a self-hosted installation, set `FI_BASE_URL` to the application API, not the OTLP collector. This example posts versioned spans to the application’s span API.

In [ ]:
%pip install -q openai==2.54.0 requests

In [ ]:
import os
from getpass import getpass

# Exported variables work in Jupyter; otherwise enter the keys at the hidden prompts.
for name in ['FI_API_KEY', 'FI_SECRET_KEY', 'OPENAI_API_KEY']:
    if not os.environ.get(name):
        value = getpass(f"{name}: ")
        if not value:
            raise ValueError(f"Missing {name}")
        os.environ[name] = value
        del value

## Define the experiment

Keep the model, question, and destination fixed. Give each prompt version its own ID, but use the same project name and `email-format` tag. The tag maps `text` to the span’s `output.value`, the actual assistant reply. A format pass does not establish whether the address is correct.

In [ ]:
import json
import time
from uuid import uuid4

import requests
from openai import OpenAI

base = os.environ.get("FI_BASE_URL", "https://api.futureagi.com").rstrip("/")
headers = {"X-Api-Key": os.environ["FI_API_KEY"], "X-Secret-Key": os.environ["FI_SECRET_KEY"]}
project = f"ridgeline-handoff-{uuid4().hex[:12]}"
question = "My tent seam tore. Who can help?"
common = "You handle support for Ridgeline Outfitters. Route damaged gear to repairs@ridgeline.example. "
tag = {"type": "OBSERVATION_SPAN_TYPE", "value": "LLM", "eval_name": "is_email",
       "custom_eval_name": "email-format", "mapping": {"text": "output.value"}, "model": "turing_flash"}

def run_version(label, instruction):
    trace_id, version_id, span_id = str(uuid4()), str(uuid4()), uuid4().hex[:16]
    messages = [{"role": "system", "content": common + instruction}, {"role": "user", "content": question}]
    with OpenAI() as client:
        start = time.time_ns()
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0)
        end = time.time_ns()
    reply = response.choices[0].message.content
    if not reply:
        raise RuntimeError("Expected a text reply")
    attrs = {"gen_ai.span.kind": "llm", "gen_ai.provider.name": "openai",
             "gen_ai.request.model": "gpt-4o-mini", "input.value": json.dumps(messages), "output.value": reply}
    if response.usage:
        attrs.update({"gen_ai.usage.input_tokens": response.usage.prompt_tokens,
                      "gen_ai.usage.output_tokens": response.usage.completion_tokens})
    span = {"trace_id": trace_id, "span_id": span_id, "name": "support.handoff",
            "project_name": project, "project_type": "experiment", "project_version_id": version_id,
            "project_version_name": label, "eval_tags": [tag], "attributes": attrs,
            "start_time": start, "end_time": end, "latency": (end - start) / 1_000_000, "status": "OK"}
    saved = requests.post(f"{base}/tracer/observation-span/create_otel_span/", headers=headers,
                          json=[span], timeout=30)
    saved.raise_for_status()
    if saved.json()["result"]["ids"] != [span_id]:
        raise RuntimeError("The API did not acknowledge the submitted span")
    print(label, reply)
    return trace_id, version_id, span_id, reply

## Run the baseline and candidate

The first instruction asks for a complete sentence. The second asks for only the mailbox. Both calls write measured timings and the returned reply, not illustrative trace data.

In [ ]:
runs = {
    "baseline": run_version("baseline", "Answer in a friendly, complete sentence with the email address."),
    "candidate": run_version("candidate", "Return only the bare email address without punctuation or explanation."),
}
print("Prototype project:", project)

## Compare the stored results

Read the same version rows shown in Prototype’s comparison table. The evaluation column is keyed by the registered tag ID, not by its display name. Each version has one scored span here. A missing value means the score is not available yet; zero is a completed failure. Check the destination too: a different mailbox can still pass a format check.

In [ ]:
# Get the project and evaluation column for this pair of versions.
_, first_version, _, _ = runs["baseline"]
response = requests.get(f"{base}/tracer/project-version/{first_version}/", headers=headers, timeout=15)
response.raise_for_status()
version = response.json()
project_id = version["project"]
[tag_info] = version["eval_tags"]
column = tag_info["custom_eval_config_id"]

wanted = {version_id for _, version_id, _, _ in runs.values()}
deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    response = requests.get(f"{base}/tracer/project-version/list_runs/", headers=headers,
                            params={"project_id": project_id}, timeout=15)
    response.raise_for_status()
    rows = {row["id"]: row for row in response.json()["result"]["table"]}
    if all(rows.get(v, {}).get(column) is not None for v in wanted):
        break
    time.sleep(2)
else:
    raise TimeoutError(f"Scores are not ready for {project}. Rerun only this comparison cell.")

for label, (trace_id, version_id, span_id, reply) in runs.items():
    row = rows[version_id]
    print(f"{row['run_name']}: email-format {row[column]:g}/100; "
          f"exact repairs address: {reply == 'repairs@ridgeline.example'}; "
          f"latency {row['avg_latency']:.0f} ms; trace {trace_id}")

## Choose a version

Open **Prototype**, select the printed project name, and compare the two replies. A candidate that returns just the repairs address satisfies this form; a polite explanation may still be a better answer in a chat window. Before using **Choose Winner**, repeat the comparison on representative cases and decide how much format, latency, and cost matter to your application. Nothing in this notebook promotes a version or changes live traffic.